<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/knnformovies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
ratings=pd.read_csv('/content/ratings.csv',sep=',',usecols=range(3),encoding='ISO-8859-1')
ratings.head()

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


In [8]:
movie_properties=ratings.groupby('movieId').agg({'rating':[np.size,np.mean]})
#movie_properties.sort_values([('rating','mean')],ascending=False).head()
movie_properties.head()

/tmp/ipykernel_16187/508530574.py:1: FutureWarning: The provided callable <function mean at 0x7b0dbfff9940> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movie_properties=ratings.groupby('movieId').agg({'rating':[np.size,np.mean]})


rating          
          size      mean
movieId                 
1          247  3.872470
2          107  3.401869
3           59  3.161017
4           13  2.384615
5           56  3.267857

In [11]:
movieNumRatings=pd.DataFrame(movie_properties['rating']['size'])
movieNumRatings=movieNumRatings.apply(lambda x: (x-np.min(x))/(np.max(x)-np.min(x)))
movieNumRatings.head()

,size
movieId,
1,0.723529
2,0.311765
3,0.170588
4,0.035294
5,0.161765


In [30]:
import csv

movieDict = {}

# Create a dictionary for genre encoding
genre_map = [ 'Action','Adventure','Animation','Children','Comedy','Crime','Drama','Fantasy','Romance']

with open('/content/movies.csv', mode='r', encoding='ISO-8859-1') as f:
    reader = csv.reader(f)
    header = next(reader)

    for fields in reader:
        if not fields:
            continue

        movieId = int(fields[0])
        name = fields[1]

        # Convert genres to integers
        genre_names = fields[2].split('|')
        genres = []
        for genre in genre_map:
            if genre in genre_names:
                genres.append(1)
            else:
                genres.append(0)




        if movieId in movieNumRatings.index and movieId in movie_properties.index:
            movieDict[movieId] = {
                'name': name,
                'genres': genres,
                'num_size': movieNumRatings.loc[movieId].get('size'),
                'mean_rating': movie_properties.loc[movieId].rating.get('mean')
            }
        else:
            movieDict[movieId] = {
                'name': name,
                'genres': genres,
                'num_size': 0,
                'mean_rating': None
            }

for k, v in list(movieDict.items())[:3]:
    print(f"ID {k}: {v}")


ID 1: {'name': 'Toy Story (1995)', 'genres': [0, 1, 1, 1, 1, 0, 0, 1, 0], 'num_size': np.float64(0.7235294117647059), 'mean_rating': np.float64(3.8724696356275303)}
ID 2: {'name': 'Jumanji (1995)', 'genres': [0, 1, 0, 1, 0, 0, 0, 1, 0], 'num_size': np.float64(0.31176470588235294), 'mean_rating': np.float64(3.4018691588785046)}
ID 3: {'name': 'Grumpier Old Men (1995)', 'genres': [0, 0, 0, 0, 1, 0, 0, 0, 1], 'num_size': np.float64(0.17058823529411765), 'mean_rating': np.float64(3.1610169491525424)}

Genre Encoding:
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Drama', 'Fantasy', 'Romance']


In [53]:
from scipy import spatial

def ComputeDistance(a, b):
    genresA = a['genres']
    genresB = b['genres']
    genreDistance=spatial.distance.cosine(genresA,genresB)
    popularityA=a['num_size']
    popularityB=b['num_size']
    popularityDistance=abs(popularityA-popularityB)
    return genreDistance+popularityDistance


In [57]:
import operator

def getNeighbors(movieID, K):
  distance=[]
  for movie,movie_details in movieDict.items():
    if(int(movie)!= movieID):
      dist=ComputeDistance(movieDict[movieID],movieDict[movie])
      distance.append((movie_details['name'],dist))
  distance.sort(key=operator.itemgetter(1))
  neighbours=[]
  for x in range(K):
    neighbours.append(distance[x][0])
  return neighbours

k=10
avgRating=0
neighbours=getNeighbors(10,k)
for neighbour in neighbours:
    print(neighbour)



Cutthroat Island (1995)
Heat (1995)
Sudden Death (1995)
Jumanji (1995)
Saturn 3 (1980)
Deliverance (1972)
Hard Day's Night, A (1964)
Assassins (1995)
Excalibur (1981)
City of Lost Children, The (CitÃ© des enfants perdus, La) (1995)
